In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import hashlib, json
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
ROOT = Path("/content/drive/MyDrive/BUSI_MERGED")
CLASSES = ('normal', 'benign', 'malignant')
EXPECTED = {'normal': 133, 'benign': 437, 'malignant': 210}
if not ROOT.is_dir():
    raise FileNotFoundError(f'Folder not found: {ROOT}. Edit ROOT if necessary.')
print('Dataset:', ROOT)

In [ ]:
def audit_dataset(root):
    rows = []
    issues = []
    counts = []

    # ثبت مشکلات برای بررسی؛ هیچ فایلی تغییر نمی‌کند.
    def issue(severity, cls, file, reason):
        issues.append({
            'severity': severity,
            'class_name': cls,
            'file': str(file),
            'reason': reason
        })

    extensions = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

    for cls in CLASSES:
        print(f'\nChecking class: {cls}', flush=True)

        # نام واقعی پوشه‌های دیتاست تو
        image_dir = root / cls / 'images'
        mask_dir = root / cls / 'masks'

        if not image_dir.is_dir() or not mask_dir.is_dir():
            issue(
                'ERROR', cls, root / cls,
                'Missing images/ or masks/ folder'
            )
            counts.append({
                'class_name': cls,
                'images': 0,
                'masks': 0,
                'expected': EXPECTED[cls]
            })
            continue

        images = sorted(
            p for p in image_dir.iterdir()
            if p.is_file() and p.suffix.lower() in extensions
        )

        masks = sorted(
            p for p in mask_dir.iterdir()
            if p.is_file() and p.suffix.lower() in extensions
        )

        print(
            f'Found {len(images)} images and {len(masks)} masks',
            flush=True
        )

        counts.append({
            'class_name': cls,
            'images': len(images),
            'masks': len(masks),
            'expected': EXPECTED[cls]
        })

        if len(images) != EXPECTED[cls]:
            issue(
                'WARNING', cls, image_dir,
                'Count differs from original BUSI; investigate preprocessing history'
            )

        used_masks = set()

        for index, ip in enumerate(images):
            # گزارش پیشرفت در اولین تصویر و سپس هر ۲۵ تصویر
            if index % 25 == 0:
                print(
                    f'  Reading {index + 1}/{len(images)}: {ip.name}',
                    flush=True
                )

            if '_mask' in ip.stem.lower():
                issue(
                    'ERROR', cls, ip,
                    'Mask-like filename inside images folder'
                )

            # مثال: benign (1).png → benign (1)_mask.png
            mp = mask_dir / (ip.stem + '_mask.png')

            row = {
                'class_name': cls,
                'image': str(ip.relative_to(root)),
                'mask': str(mp.relative_to(root)),
                'width': None,
                'height': None,
                'image_hash': None,
                'mask_values': None,
                'foreground_fraction': None,
                'pair_ok': False
            }

            # خواندن تصویر و محاسبه شناسه پیکسل‌های آن
            try:
                with Image.open(ip) as im:
                    rgb = np.array(im.convert('RGB'))

                row['height'], row['width'] = rgb.shape[:2]

                row['image_hash'] = hashlib.sha256(
                    str(rgb.shape).encode() + rgb.tobytes()
                ).hexdigest()

            except Exception as exc:
                issue(
                    'ERROR', cls, ip,
                    f'Cannot decode image: {exc}'
                )
                rows.append(row)
                continue

            # بررسی وجود ماسک متناظر
            if not mp.is_file():
                issue(
                    'ERROR', cls, ip,
                    'Matching *_mask.png is missing'
                )
                rows.append(row)
                continue

            used_masks.add(mp)

            try:
                with Image.open(mp) as mask_im:
                    raw = np.array(mask_im)
                    mask = np.array(mask_im.convert('L'))

                # ماسک رنگی با کانال‌های متفاوت نیاز به بررسی دارد.
                if raw.ndim == 3 and raw.shape[2] >= 3:
                    channels_equal = (
                        np.array_equal(raw[..., 0], raw[..., 1])
                        and np.array_equal(raw[..., 1], raw[..., 2])
                    )

                    if not channels_equal:
                        issue(
                            'ERROR', cls, mp,
                            'Mask has unequal RGB channels; inspect color-label encoding'
                        )

                values = np.unique(mask)
                row['mask_values'] = json.dumps(values.tolist())

                # سهم پیکسل‌های غیرصفر؛ فقط برای بررسی اولیه
                row['foreground_fraction'] = float((mask > 0).mean())

                same_size = mask.shape == rgb.shape[:2]

                if not same_size:
                    issue(
                        'ERROR', cls, mp,
                        'Image and mask sizes differ'
                    )

                binary_values = (
                    set(values.tolist()) <= {0, 1}
                    or set(values.tolist()) <= {0, 255}
                )

                if not binary_values:
                    issue(
                        'WARNING', cls, mp,
                        'Nonbinary mask values; inspect before deciding a threshold'
                    )

                if cls == 'normal' and np.any(mask > 0):
                    issue(
                        'ERROR', cls, mp,
                        'Normal mask is not empty'
                    )

                if cls != 'normal' and not np.any(mask > 0):
                    issue(
                        'ERROR', cls, mp,
                        'Lesion class has an empty mask'
                    )

                # فقط یعنی تصویر و ماسک خوانده شدند و اندازه یکسان دارند.
                # صحت کامل ماسک را باید از جدول مشکلات و نمایش تصویری بررسی کرد.
                row['pair_ok'] = same_size

            except Exception as exc:
                issue(
                    'ERROR', cls, mp,
                    f'Cannot decode mask: {exc}'
                )

            rows.append(row)

        # ماسک‌هایی که هیچ تصویر متناظری ندارند
        for mp in masks:
            if mp not in used_masks:
                issue(
                    'ERROR', cls, mp,
                    'Orphan / extra mask without a matched image'
                )

        print(f'Finished class: {cls}', flush=True)

    print('\nChecking exact duplicate images...', flush=True)

    groups = defaultdict(list)

    for row in rows:
        if row['image_hash']:
            groups[row['image_hash']].append(row)

    duplicate_rows = []
    group_number = 0

    for digest, members in sorted(groups.items()):
        if len(members) < 2:
            continue

        group_number += 1

        # یک تصویر یکسان با برچسب‌های متفاوت، تعارض برچسب دارد.
        conflict = len({r['class_name'] for r in members}) > 1

        for r in members:
            duplicate_rows.append({
                'group_id': group_number,
                'image_hash': digest,
                'class_name': r['class_name'],
                'image': r['image'],
                'cross_class_conflict': conflict
            })

            issue(
                'ERROR' if conflict else 'WARNING',
                r['class_name'],
                r['image'],
                (
                    'Exact image duplicate across classes'
                    if conflict
                    else 'Exact image duplicate; keep group in a single split later'
                )
            )

    return (
        pd.DataFrame(rows, columns=[
            'class_name', 'image', 'mask', 'width', 'height',
            'image_hash', 'mask_values',
            'foreground_fraction', 'pair_ok'
        ]),
        pd.DataFrame(issues, columns=[
            'severity', 'class_name', 'file', 'reason'
        ]),
        pd.DataFrame(counts, columns=[
            'class_name', 'images', 'masks', 'expected'
        ]),
        pd.DataFrame(duplicate_rows, columns=[
            'group_id', 'image_hash', 'class_name',
            'image', 'cross_class_conflict'
        ])
    )


# اجرای تابع و دریافت چهار جدول
manifest, issues, counts, duplicates = audit_dataset(ROOT)

print('\nCLASS COUNTS')
display(counts)

errors = int((issues['severity'] == 'ERROR').sum())
warnings = int((issues['severity'] == 'WARNING').sum())

summary = {
    'total_images': len(manifest),
    'errors': errors,
    'warnings': warnings,
    'exact_duplicate_groups': int(duplicates['group_id'].nunique()),
    'root': str(ROOT),
    'checked_at_utc': datetime.now(timezone.utc).isoformat()
}

print(json.dumps(summary, indent=2))

print('ERROR: investigate before training.')
print('WARNING: review; it does not necessarily mean the file is unusable.')
print('This audit does not rule out near-duplicates or patient leakage.')

if not issues.empty:
    print('\nISSUES — first 40 rows')
    display(issues.head(40))

if not duplicates.empty:
    print('\nEXACT DUPLICATES — first 30 rows')
    display(duplicates.head(30))

In [ ]:
rng = np.random.default_rng(42)
for cls in CLASSES:
    valid = clean_manifest[
    (clean_manifest['class_name'] == cls)
    & clean_manifest['pair_ok']
]
    if valid.empty:
        print('No readable matched pairs for:', cls)
        continue
    chosen = valid.iloc[rng.choice(len(valid), size=min(3, len(valid)), replace=False)]
    fig, axes = plt.subplots(len(chosen), 3, figsize=(12, 3.5 * len(chosen)), squeeze=False)
    for idx, (_, row) in enumerate(chosen.iterrows()):
        with Image.open(ROOT / row['image']) as im:
            img = np.array(im.convert('RGB'))
        with Image.open(ROOT / row['mask']) as im:
            mask = np.array(im.convert('L')) > 0
        axes[idx,0].imshow(img)
        axes[idx,0].set_title(row['image'], fontsize=9)
        axes[idx,1].imshow(mask, cmap='gray', vmin=0, vmax=1)
        axes[idx,1].set_title('Mask: foreground > 0 (audit display only)')
        axes[idx,2].imshow(img)
        overlay = np.zeros((*mask.shape, 4))
        overlay[mask] = (1, 0, 0, 0.4)
        axes[idx,2].imshow(overlay)
        axes[idx,2].set_title('Overlay')
        for ax in axes[idx]: ax.axis('off')
    fig.suptitle(cls)
    plt.tight_layout()
    plt.show()

In [ ]:
report_dir = ROOT.parent / 'BUSI_AUDIT_REPORTS' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
report_dir.mkdir(parents=True, exist_ok=False)
manifest.to_csv(report_dir / 'manifest.csv', index=False)
issues.to_csv(report_dir / 'issues.csv', index=False)
counts.to_csv(report_dir / 'class_counts.csv', index=False)
duplicates.to_csv(report_dir / 'exact_duplicates.csv', index=False)
(report_dir / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Reports saved:', report_dir)

In [ ]:
pairs = [
    ('benign', 'benign (433)'),
    ('malignant', 'malignant (145)')
]

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for row, (cls, name) in enumerate(pairs):
    image_path = ROOT / cls / 'images' / f'{name}.png'
    mask_path = ROOT / cls / 'masks' / f'{name}_mask.png'

    with Image.open(image_path) as im:
        image = np.array(im.convert('RGB'))

    with Image.open(mask_path) as im:
        mask = np.array(im.convert('L')) > 0

    axes[row, 0].imshow(image)
    axes[row, 0].set_title(name)

    axes[row, 1].imshow(mask, cmap='gray', vmin=0, vmax=1)
    axes[row, 1].set_title('Mask')

    axes[row, 2].imshow(image)
    axes[row, 2].imshow(
        np.ma.masked_where(~mask, mask),
        cmap='autumn', alpha=0.4, vmin=0, vmax=1
    )
    axes[row, 2].set_title('Overlay')

    for ax in axes[row]:
        ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# نمونه‌های دارای برچسب متناقض
excluded_images = [
    'benign/images/benign (433).png',
    'malignant/images/malignant (145).png'
]

# نگهداری فهرست نمونه‌های کنارگذاشته‌شده
excluded_manifest = manifest[
    manifest['image'].isin(excluded_images)
].copy()

excluded_manifest['exclusion_reason'] = (
    'Identical image with conflicting class labels'
)

# فهرست قابل استفاده؛ فایل‌های اصلی حذف نمی‌شوند.
clean_manifest = manifest[
    ~manifest['image'].isin(excluded_images)
].copy()

print('Original images:', len(manifest))
print('Excluded images:', len(excluded_manifest))
print('Remaining images:', len(clean_manifest))

display(
    clean_manifest.groupby('class_name')
    .size()
    .rename('images')
    .reset_index()
)

In [ ]:
# پوشهٔ گزارش همین اجرا که قبلاً ساخته شده است
clean_manifest.to_csv(
    report_dir / 'clean_manifest.csv',
    index=False
)

excluded_manifest.to_csv(
    report_dir / 'excluded_manifest.csv',
    index=False
)

print('فهرست داده و دلیل کنارگذاری ذخیره شد.')


In [ ]:
fingerprints = []

for index, row in enumerate(clean_manifest.itertuples()):
    image_path = ROOT / row.image

    with Image.open(image_path) as im:
        small = im.convert('L').resize(
            (17, 16),
            Image.Resampling.LANCZOS
        )

    pixels = np.array(small, dtype=np.int16)

    # مقایسه روشنایی هر پیکسل با همسایه سمت چپ
    fingerprint = pixels[:, 1:] > pixels[:, :-1]

    fingerprints.append(fingerprint.flatten())

    if index % 100 == 0:
        print(f'Processed {index + 1}/{len(clean_manifest)}')

fingerprints = np.array(fingerprints)

print('Fingerprint shape:', fingerprints.shape)

In [ ]:
data = clean_manifest.reset_index(drop=True)
candidates = []

# آستانه اولیه برای پیشنهاد جفت‌ها؛ معیار قطعی تکراری‌بودن نیست.
MAX_DIFFERENCES = 8

for i in range(len(data) - 1):
    differences = np.count_nonzero(
        fingerprints[i + 1:] != fingerprints[i],
        axis=1
    )

    for offset in np.where(differences <= MAX_DIFFERENCES)[0]:
        j = i + 1 + int(offset)

        candidates.append({
            'image_1': data.loc[i, 'image'],
            'image_2': data.loc[j, 'image'],
            'class_1': data.loc[i, 'class_name'],
            'class_2': data.loc[j, 'class_name'],
            'differences': int(differences[offset])
        })

similar_pairs = pd.DataFrame(candidates, columns=[
    'image_1', 'image_2', 'class_1', 'class_2', 'differences'
]).sort_values('differences').reset_index(drop=True)

print('Candidate pairs:', len(similar_pairs))
display(similar_pairs.head(15))

similar_pairs.to_csv(
    report_dir / 'similarity_candidates.csv',
    index=False
)

In [ ]:
# اول جفت‌هایی که برچسب متفاوت دارند
review_pairs = similar_pairs.copy()

review_pairs['cross_class'] = (
    review_pairs['class_1'] != review_pairs['class_2']
)

review_pairs = review_pairs.sort_values(
    ['cross_class', 'differences'],
    ascending=[False, True]
).reset_index(drop=True)

print('Total candidate pairs:', len(review_pairs))
print('Cross-class pairs:', int(review_pairs['cross_class'].sum()))

# برای نمایش دسته بعدی، این عدد را به 6، سپس 12 و ... تغییر بده.
START =  3
batch = review_pairs.iloc[START:START + 1]

if batch.empty:
    print('No pairs in this batch.')
else:
    fig, axes = plt.subplots(
        len(batch), 2,
        figsize=(10, 3.5 * len(batch)),
        squeeze=False
    )

    for row_number, (pair_id, pair) in enumerate(batch.iterrows()):
        for column, key in enumerate(['image_1', 'image_2']):
            with Image.open(ROOT / pair[key]) as im:
                axes[row_number, column].imshow(im.convert('RGB'))

            axes[row_number, column].set_title(
                f"Pair {pair_id} | {pair[key]}\n"
                f"Fingerprint differences: {pair['differences']}",
                fontsize=9
            )
            axes[row_number, column].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# دو نمونه جدید با تعارض برچسب مشکوک
new_exclusions = [
    'benign/images/benign (131).png',
    'malignant/images/malignant (51).png'
]

new_rows = manifest[
    manifest['image'].isin(new_exclusions)
].copy()

assert len(new_rows) == 2, 'نام یا مسیر دو فایل را بررسی کن.'

new_rows['exclusion_reason'] = (
    'Visually near-identical images with conflicting labels; '
    'excluded pending label verification'
)

# اضافه‌کردن به کنارگذاری‌های قبلی، بدون تکرار ردیف‌ها
excluded_manifest = pd.concat(
    [excluded_manifest, new_rows],
    ignore_index=True
).drop_duplicates(subset='image')

clean_manifest = manifest[
    ~manifest['image'].isin(excluded_manifest['image'])
].copy()

# ذخیره تصمیم‌ها
excluded_manifest.to_csv(
    report_dir / 'excluded_manifest.csv', index=False
)
clean_manifest.to_csv(
    report_dir / 'clean_manifest.csv', index=False
)

print('Remaining images:', len(clean_manifest))

display(
    clean_manifest.groupby('class_name')
    .size()
    .rename('images')
    .reset_index()
)

In [ ]:
# جدول بررسی‌ها را فقط بار اول می‌سازیم.
if 'reviewed_pairs' not in globals():
    reviewed_pairs = pd.DataFrame()

confirmed = review_pairs.loc[30:35].copy()
confirmed['decision'] = 'keep_in_same_split'
confirmed['review_basis'] = 'visual_review'

reviewed_pairs = pd.concat(
    [reviewed_pairs, confirmed],
    ignore_index=True
).drop_duplicates(subset=['image_1', 'image_2'])

reviewed_pairs.to_csv(
    report_dir / 'reviewed_pairs.csv',
    index=False
)

print('این شش جفت ثبت شدند.')

In [ ]:
# جفت صفر کنار گذاشته شده؛ جفت ۳ هنوز کامل بررسی نشده است.
confirmed = review_pairs.loc[[1, 2, 4, 5]].copy()
confirmed['decision'] = 'keep_in_same_split'
confirmed['review_basis'] = 'visual_review'

reviewed_pairs = pd.concat(
    [reviewed_pairs, confirmed],
    ignore_index=True
).drop_duplicates(subset=['image_1', 'image_2'])

reviewed_pairs.to_csv(
    report_dir / 'reviewed_pairs.csv',
    index=False
)

print('Registered pairs:', len(reviewed_pairs))
display(review_pairs.loc[[3], ['image_1', 'image_2']])

In [ ]:
confirmed = review_pairs.loc[[3]].copy()
confirmed['decision'] = 'keep_in_same_split'
confirmed['review_basis'] = 'visual_review'

reviewed_pairs = pd.concat(
    [reviewed_pairs, confirmed],
    ignore_index=True
).drop_duplicates(subset=['image_1', 'image_2'])

reviewed_pairs.to_csv(
    report_dir / 'reviewed_pairs.csv',
    index=False
)

print('Registered pairs:', len(reviewed_pairs))

In [ ]:
assert len(reviewed_pairs) == 35, 'ثبت جفت‌ها هنوز کامل نیست.'

# در ابتدا، هر تصویر گروه مستقل خودش را دارد.
parent = {
    image: image
    for image in clean_manifest['image']
}

# پیدا کردن نمایندهٔ گروه یک تصویر
def find(image):
    while parent[image] != image:
        parent[image] = parent[parent[image]]
        image = parent[image]
    return image

# اتصال گروه‌ها بر اساس جفت‌های بررسی‌شده
for pair in reviewed_pairs.itertuples():
    a, b = pair.image_1, pair.image_2

    assert a in parent and b in parent, 'نمونه‌ای خارج از فهرست فعلی است.'

    root_a = find(a)
    root_b = find(b)

    if root_a != root_b:
        # انتخاب ثابت نماینده برای قابل‌تکراربودن گروه‌بندی
        first, second = sorted([root_a, root_b])
        parent[second] = first

# اختصاص شناسه به هر گروه
representatives = sorted({
    find(image) for image in parent
})

group_ids = {
    representative: f'group_{index:04d}'
    for index, representative in enumerate(representatives)
}

grouped_manifest = clean_manifest.copy()

grouped_manifest['group_id'] = [
    group_ids[find(image)]
    for image in grouped_manifest['image']
]

# گروه‌ها در این فهرست نباید برچسب‌های متناقض داشته باشند.
assert (
    grouped_manifest.groupby('group_id')['class_name']
    .nunique()
    .eq(1)
    .all()
), 'یک گروه دارای برچسب‌های متفاوت است.'

grouped_manifest.to_csv(
    report_dir / 'grouped_manifest.csv',
    index=False
)

group_sizes = grouped_manifest.groupby('group_id').size()

print('Images:', len(grouped_manifest))
print('Groups:', len(group_sizes))
print('Groups with multiple images:', int((group_sizes > 1).sum()))
print('Largest group:', int(group_sizes.max()))

In [ ]:
from pathlib import Path
import pandas as pd

# محل دیتاست در گوگل‌درایو
ROOT = Path("/content/drive/MyDrive/BUSI_MERGED")

# تصاویر دارای تعارض برچسب در گزارش منتشرشده
reported_cases = {
    "benign": [
        42, 85, 131, 164, 269, 333,
        399, 406, 433, 437
    ],
    "malignant": [
        51, 52, 77, 78, 93, 94, 145
    ],
    "normal": [1, 34]
}

rows = []

for class_name, numbers in reported_cases.items():
    for number in numbers:
        # مثلاً: benign/images/benign (42).png
        relative_path = (
            f"{class_name}/images/{class_name} ({number}).png"
        )

        rows.append({
            "class_name": class_name,
            "image": relative_path,
            "exists": (ROOT / relative_path).is_file()
        })

conflict_check = pd.DataFrame(rows)

display(conflict_check)
print("Reported images:", len(conflict_check))
print("Found images:", int(conflict_check["exists"].sum()))

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

ROOT = Path("/content/drive/MyDrive/BUSI_MERGED")

# هر گروه شامل تصاویر مشابه با برچسب‌های متفاوت است.
conflict_groups = [
    [("benign", 42), ("benign", 131), ("malignant", 51)],
    [("benign", 85), ("benign", 164), ("malignant", 52)],
    [("benign", 333), ("malignant", 77), ("malignant", 78)],
    [("benign", 399), ("malignant", 93)],
    [("benign", 406), ("malignant", 94)],
    [("benign", 433), ("malignant", 145)],
    [("benign", 269), ("normal", 1)],
    [("benign", 437), ("normal", 34)],
]

for group_number, group in enumerate(conflict_groups, start=1):

    # برای هر گروه یک ردیف تصویر می‌سازیم.
    fig, axes = plt.subplots(
        1, len(group),
        figsize=(5 * len(group), 4)
    )

    for ax, (class_name, number) in zip(axes, group):
        image_path = (
            ROOT / class_name / "images"
            / f"{class_name} ({number}).png"
        )

        with Image.open(image_path) as image:
            ax.imshow(image.convert("L"), cmap="gray")

        ax.set_title(f"{class_name} ({number})")
        ax.axis("off")

    fig.suptitle(f"Reported conflict group {group_number}")
    plt.tight_layout()
    plt.show()

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

# باید از فهرست اولیهٔ ۷۸۰ تصویر شروع کنیم.
assert len(manifest) == 780, (
    "manifest باید فهرست اولیهٔ ۷۸۰ تصویر باشد."
)
assert manifest["image"].is_unique, "مسیر تصویر تکراری است."

# نام فایل‌های ۱۹ تصویر، از گروه‌هایی که بررسی کردی
conflict_images = {
    f"{class_name}/images/{class_name} ({number}).png"
    for group in conflict_groups
    for class_name, number in group
}

assert len(conflict_images) == 19

# True یعنی این تصویر جزو موارد تعارض است.
is_conflict = manifest["image"].isin(conflict_images)

# فهرست کنارگذاشته‌شده‌ها، همراه دلیل
excluded_manifest_v2 = manifest.loc[is_conflict].copy()
excluded_manifest_v2["reason"] = (
    "Published BUSI cross-class duplicate; "
    "visual similarity confirmed by user; label unresolved"
)

# فهرست تصاویر باقی‌مانده
clean_manifest_v2 = (
    manifest.loc[~is_conflict]
    .copy()
    .reset_index(drop=True)
)

assert len(excluded_manifest_v2) == 19
assert len(clean_manifest_v2) == 761

# ذخیرهٔ نسخهٔ جدید در پوشه‌ای جدا
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
report_dir_v2 = Path(
    "/content/drive/MyDrive/BUSI_AUDIT_REPORTS"
) / f"revision_{timestamp}"

report_dir_v2.mkdir(parents=True, exist_ok=True)

clean_manifest_v2.to_csv(
    report_dir_v2 / "clean_manifest.csv", index=False
)
excluded_manifest_v2.to_csv(
    report_dir_v2 / "excluded_manifest.csv", index=False
)

print("Original images:", len(manifest))
print("Excluded images:", len(excluded_manifest_v2))
print("Remaining images:", len(clean_manifest_v2))

display(
    clean_manifest_v2.groupby("class_name")
    .size()
    .rename("images")
    .reset_index()
)

print("Reports saved:", report_dir_v2)

In [ ]:
!pip -q install pypdf

In [ ]:
import requests
import re
import pandas as pd
from pypdf import PdfReader

# دانلود گزارش علمی
url = "https://www.ippt.pan.pl/repository/open/o8361.pdf"
response = requests.get(url, timeout=60)
response.raise_for_status()

pdf_path = report_dir_v2 / "BUSI_published_review.pdf"
pdf_path.write_bytes(response.content)

# استخراج متن صفحات PDF
reader = PdfReader(str(pdf_path))
text = "\n".join(
    page.extract_text() or ""
    for page in reader.pages
)

# جدول B1 شامل زوج‌های تکراری است.
# آخرین محل عنوان را می‌گیریم تا ارجاع داخل متن انتخاب نشود.
start = text.rfind("Table B1")
end = text.rfind("Table B2")

assert 0 <= start < end, "عنوان جدول‌ها پیدا نشد."

table_text = text[start:end]

# استخراج زوج‌هایی مثل {1, 318}
number_pairs = re.findall(
    r"\{\s*(\d+)\s*,\s*(\d+)\s*\}",
    table_text
)

number_pairs = sorted({
    (int(a), int(b))
    for a, b in number_pairs
})

assert len(number_pairs) > 100, (
    "استخراج جدول ناقص به نظر می‌رسد؛ ادامه نده."
)
assert all(
    1 <= a <= 780 and 1 <= b <= 780
    for a, b in number_pairs
)

# گزارش شماره‌های هر سه کلاس را پشت سر هم نوشته است.
# این تابع شمارهٔ گزارش را به مسیر واقعی فایل تبدیل می‌کند.
def report_id_to_path(number):
    if number <= 437:
        class_name = "benign"
        file_number = number
    elif number <= 647:
        class_name = "malignant"
        file_number = number - 437
    else:
        class_name = "normal"
        file_number = number - 647

    return (
        f"{class_name}/images/"
        f"{class_name} ({file_number}).png"
    )

published_pairs = pd.DataFrame([
    {
        "image_1": report_id_to_path(a),
        "image_2": report_id_to_path(b)
    }
    for a, b in number_pairs
])

# اطمینان از اینکه مسیرها در فهرست اولیه وجود دارند
original_paths = set(manifest["image"])

assert set(published_pairs["image_1"]).issubset(original_paths)
assert set(published_pairs["image_2"]).issubset(original_paths)

published_pairs.to_csv(
    report_dir_v2 / "published_duplicate_pairs.csv",
    index=False
)

print("Extracted pairs:", len(published_pairs))
display(published_pairs.head(10))

In [ ]:
import pandas as pd

# تصاویر مجاز، پس از کنارگذاشتن ۱۹ مورد تعارض
valid_images = set(clean_manifest_v2["image"])

# ترکیب فهرست مقاله با زوج‌هایی که قبلاً خودت بررسی کردی
all_pairs = pd.concat(
    [
        published_pairs[["image_1", "image_2"]],
        reviewed_pairs[["image_1", "image_2"]]
    ],
    ignore_index=True
).drop_duplicates()

# فقط زوج‌هایی را نگه می‌داریم که هر دو تصویرشان باقی مانده‌اند.
usable_pairs = all_pairs[
    all_pairs["image_1"].isin(valid_images)
    & all_pairs["image_2"].isin(valid_images)
].copy()

# ابتدا هر تصویر نمایندهٔ گروه خودش است.
parent = {image: image for image in valid_images}

def find(image):
    """نمایندهٔ نهایی گروه تصویر را پیدا می‌کند."""
    while parent[image] != image:
        parent[image] = parent[parent[image]]
        image = parent[image]
    return image

def union(image_1, image_2):
    """گروه دو تصویر مشابه را به هم وصل می‌کند."""
    root_1 = find(image_1)
    root_2 = find(image_2)

    if root_1 != root_2:
        smaller, larger = sorted([root_1, root_2])
        parent[larger] = smaller

# اتصال تصاویر بر اساس زوج‌های مشابه
for row in usable_pairs.itertuples(index=False):
    union(row.image_1, row.image_2)

# اتصال تکرارهای دقیق ثبت‌شده در بررسی اولیه، در صورت وجود
for _, subset in clean_manifest_v2.dropna(
    subset=["image_hash"]
).groupby("image_hash"):
    images = subset["image"].tolist()
    for image in images[1:]:
        union(images[0], image)

# ساخت نام ثابت برای گروه‌ها
roots = sorted({find(image) for image in valid_images})
group_names = {
    root: f"group_{number:04d}"
    for number, root in enumerate(roots)
}

grouped_manifest_v2 = clean_manifest_v2.copy()
grouped_manifest_v2["group_id"] = (
    grouped_manifest_v2["image"]
    .map(lambda image: group_names[find(image)])
)

# پس از حذف تعارض‌ها، هر گروه باید فقط یک کلاس داشته باشد.
classes_per_group = (
    grouped_manifest_v2.groupby("group_id")["class_name"]
    .nunique()
)
assert classes_per_group.max() == 1, (
    "هنوز یک گروه با برچسب‌های متناقض وجود دارد."
)
assert len(grouped_manifest_v2) == 761

group_sizes = grouped_manifest_v2.groupby("group_id").size()

grouped_manifest_v2.to_csv(
    report_dir_v2 / "grouped_manifest.csv", index=False
)
usable_pairs.to_csv(
    report_dir_v2 / "used_duplicate_pairs.csv", index=False
)

print("Images:", len(grouped_manifest_v2))
print("Groups:", len(group_sizes))
print("Groups with multiple images:", int((group_sizes > 1).sum()))
print("Largest group:", int(group_sizes.max()))

In [ ]:
from sklearn.model_selection import train_test_split
import json

SEED = 42

# یک ردیف برای هر گروه؛ هر گروه فقط یک کلاس دارد.
group_table = (
    grouped_manifest_v2[["group_id", "class_name"]]
    .drop_duplicates()
    .sort_values("group_id")
    .reset_index(drop=True)
)

assert group_table["group_id"].is_unique

# ابتدا ۲۰٪ گروه‌ها را برای آزمون جدا می‌کنیم.
train_val_groups, test_groups = train_test_split(
    group_table,
    test_size=0.20,
    stratify=group_table["class_name"],
    random_state=SEED
)

# سپس ۲۰٪ بخش باقی‌مانده برای اعتبارسنجی:
# ۰٫۲۰ × ۰٫۸۰ = ۰٫۱۶ کل گروه‌ها
train_groups, val_groups = train_test_split(
    train_val_groups,
    test_size=0.20,
    stratify=train_val_groups["class_name"],
    random_state=SEED
)

# تعیین بخش مربوط به هر گروه
split_by_group = {}

for split_name, table in [
    ("train", train_groups),
    ("val", val_groups),
    ("test", test_groups)
]:
    for group_id in table["group_id"]:
        split_by_group[group_id] = split_name

# هر تصویر، بخش گروه خودش را می‌گیرد.
split_manifest_v2 = grouped_manifest_v2.copy()
split_manifest_v2["split"] = (
    split_manifest_v2["group_id"].map(split_by_group)
)

# بررسی کامل‌بودن تقسیم
assert len(split_manifest_v2) == 761
assert split_manifest_v2["image"].is_unique
assert split_manifest_v2["split"].notna().all()

# هیچ گروهی نباید در بیش از یک بخش حضور داشته باشد.
assert (
    split_manifest_v2.groupby("group_id")["split"]
    .nunique()
    .max()
) == 1

# جدول تعداد تصاویر در هر بخش و کلاس
counts = pd.crosstab(
    split_manifest_v2["split"],
    split_manifest_v2["class_name"]
).reindex(["train", "val", "test"])

counts["total"] = counts.sum(axis=1)
display(counts)

print("Image percentages:")
print((100 * counts["total"] / 761).round(2))

# ذخیرهٔ تقسیم جدید
split_path_v2 = report_dir_v2 / "split_manifest.csv"
split_manifest_v2.to_csv(split_path_v2, index=False)

settings = {
    "seed": SEED,
    "images": len(split_manifest_v2),
    "groups": len(group_table),
    "target_group_ratios": {
        "train": 0.64,
        "val": 0.16,
        "test": 0.20
    },
    "excluded_cross_class_images": 19,
    "grouping_sources": [
        "Published BUSI review, Table B1",
        "Previously reviewed similar pairs",
        "Exact decoded-image hashes"
    ],
    "patient_independence_verified": False
}

with open(
    report_dir_v2 / "split_settings.json",
    "w", encoding="utf-8"
) as file:
    json.dump(settings, file, ensure_ascii=False, indent=2)

print("New split saved:", split_path_v2)